# Quickstart: BPTI Conformation Generation

This notebook runs the released model on the small BPTI example included in the repository.

It uses the same command-line sampler described in the README, then loads the generated PDB so you can quickly inspect the result.

## 1. User Settings

Edit the values in the next cell before running the notebook.

The default checkpoint path follows the README. If you downloaded the checkpoint somewhere else, change `CHECKPOINT_PATH` in the next cell.

If you change `CUDA_VISIBLE_DEVICES` after importing `torch`, restart the notebook kernel before running again.


In [ ]:
from pathlib import Path
import os

# Edit these values first.
CHECKPOINT_PATH = Path("/data/hier_ConfGen_ckpt/mp_rank_00_model_states.pt")
# If you saved the checkpoint somewhere else, update this path.

# A relative output path is interpreted under hierarchical_ConfGen/.
OUTPUT_DIR = Path("outputs/bpti_notebook_demo")

# Set this to an idle GPU on a shared server, for example "2". Use None to keep the current environment.
CUDA_VISIBLE_DEVICES = None

# Use the included BPTI example by default. Set INPUT_DIR to your own folder of .pdb files if needed.
INPUT_DIR = None

if CUDA_VISIBLE_DEVICES is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)

repo_root = Path.cwd().resolve()
if not (repo_root / "hierarchical_ConfGen").is_dir():
    if (repo_root.parent / "hierarchical_ConfGen").is_dir():
        repo_root = repo_root.parent.resolve()
    else:
        raise RuntimeError("Could not find hierarchical_ConfGen. Please run this notebook from the repository root.")

confgen_dir = repo_root / "hierarchical_ConfGen"
input_dir = Path(INPUT_DIR) if INPUT_DIR is not None else confgen_dir / "data" / "targets" / "bpti"
output_dir = Path(OUTPUT_DIR)
if not output_dir.is_absolute():
    output_dir = confgen_dir / output_dir
checkpoint_path = Path(CHECKPOINT_PATH)

print("Repository:", repo_root)
print("Input directory:", input_dir)
print("Output directory:", output_dir)
print("Checkpoint:", checkpoint_path)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "not set"))


## 2. Environment Check

This should print `cuda: True` on a GPU machine. The example can be made smaller with fewer samples, but the released model is intended to run with CUDA.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Check Input and Checkpoint Files

The input is a directory of PDB files. Here we use the included BPTI example.

If the checkpoint is missing, download it from Zenodo as described in the README.

In [ ]:
input_pdb = input_dir / "bpti.pdb"

checks = {
    "input directory": input_dir,
    "BPTI input PDB": input_pdb,
    "diffusion checkpoint": checkpoint_path,
}

missing = []
for name, path in checks.items():
    exists = path.exists()
    print(f"{name}: {path}  {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append((name, path))

if missing:
    details = "\n".join(f"- {name}: {path}" for name, path in missing)
    raise FileNotFoundError(
        "Missing required file(s):\n"
        f"{details}\n\n"
        "If the checkpoint was downloaded to a different location, update CHECKPOINT_PATH "
        "in the first code cell and run the notebook again."
    )


## 4. Run a Small Inference Job

This first run uses only 2 samples and 5 denoising steps. It is meant to check that the full path works before launching a larger job.

In [ ]:
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = f"{confgen_dir}:{env.get('PYTHONPATH', '')}"

cmd = [
    sys.executable,
    "slm/sample_esmdiff_var.py",
    "--input", "data/targets/bpti",
    "--output", str(output_dir),
    "--num_steps", "5",
    "--num_samples", "2",
    "--ckpt", str(checkpoint_path),
]

print("Running:")
print(" ".join(cmd))

subprocess.run(cmd, cwd=confgen_dir, env=env, check=True)

## 5. Find the Generated Ensemble

The sampler creates a timestamped folder under the output directory. The generated PDB is a multi-model ensemble.

In [ ]:
generated_pdbs = sorted(output_dir.rglob("*.pdb"), key=lambda p: p.stat().st_mtime, reverse=True)
if not generated_pdbs:
    raise FileNotFoundError(f"No generated PDB files found under {output_dir}")

generated_pdb = generated_pdbs[0]
text = generated_pdb.read_text(errors="replace")
n_models = text.count("\nMODEL") + (1 if text.startswith("MODEL") else 0)

print("Generated PDB:", generated_pdb)
print("Number of MODEL records:", n_models if n_models else "not explicitly marked")
print("File size:", f"{generated_pdb.stat().st_size / 1024:.1f} KB")

## 6. Visualize the Input and Output

The cells below use `py3Dmol`. If it is missing, install it in the active environment:

```bash
pip install py3Dmol
```

In [ ]:
try:
    import py3Dmol
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install py3Dmol with: pip install py3Dmol") from exc

def show_pdb(path, width=720, height=520):
    pdb_text = Path(path).read_text(errors="replace")
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_text, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view

Input structure:

In [ ]:
show_pdb(input_pdb)

Generated ensemble:

In [ ]:
show_pdb(generated_pdb)

## Larger Runs

After the small example works, increase the values in the command cell or run from the command line:

```bash
cd /hierarchical_conformation_generation/hierarchical_ConfGen
export PYTHONPATH=/hierarchical_conformation_generation/hierarchical_ConfGen:$PYTHONPATH

CUDA_VISIBLE_DEVICES=0 python slm/sample_esmdiff_var.py \
  --input data/targets/bpti \
  --output outputs/bpti \
  --num_steps 25 \
  --num_samples 100 \
  --ckpt /data/hier_ConfGen_ckpt/mp_rank_00_model_states.pt
```

For your own protein, put one or more `.pdb` files in a folder and pass that folder to `--input`, or set `INPUT_DIR` in the first code cell.
